# CBOW and Skip-Gram — Word2Vec from Scratch

In this notebook we implement both CBOW and Skip-Gram from scratch using only NumPy. No external NLP libraries.

- **CBOW** (Continuous Bag of Words): given surrounding context words, predict the center word
- **Skip-Gram**: given the center word, predict each surrounding context word

Both learn word vectors (embeddings) as a by-product of this prediction task.

In [ ]:
# Only standard libraries needed — no gensim for this notebook
import numpy as np
import re

print("Libraries imported. Ready to build Word2Vec from scratch.")

Libraries imported. Ready to build Word2Vec from scratch.


## Step 1: Text Corpus

We use a small English text about computer vision (YOLO framework). In real projects, a corpus of millions of sentences would be used.

In [ ]:
# Text corpus about computer vision and YOLO framework
raw_text = """
Ultralytics YOLO is a versatile AI framework that supports multiple computer vision tasks.
The framework can be used to perform detection, segmentation, classification, and pose estimation.
Detection involves identifying objects in an image and drawing bounding boxes around them.
The detected objects are classified into different categories based on their visual features.
YOLO can detect multiple objects in a single image or video frame with high speed.
Segmentation takes object detection further by producing pixel level masks for each detected object.
This precision is useful for applications such as medical imaging and manufacturing quality control.
Classification involves categorizing entire images based on their visual content and patterns.
Pose estimation detects specific keypoints in images to track movements and estimate body poses.
These keypoints can represent human joints, facial features, or other significant interest points.
YOLO excels at keypoint detection with high accuracy making it valuable for sports analytics.
Object detection is essential for autonomous vehicles and real time surveillance systems.
Image segmentation produces a precise map of every pixel belonging to each detected object class.
Computer vision models learn features from large datasets of labeled images during training.
"""

print(f"Corpus loaded: {len(raw_text)} characters")

Corpus loaded: 1327 characters


## Step 2: Preprocessing

Convert to lowercase, strip punctuation, tokenize.

In [ ]:
def preprocess(text):
    # Convert all text to lowercase
    text = text.lower()
    # Remove anything that is not a letter or space
    text = re.sub(r'[^a-z\s]', '', text)
    sentences = []
    for line in text.strip().split('\n'):
        words = line.strip().split()
        # Only keep lines that have at least 2 words
        if len(words) > 1:
            sentences.append(words)
    return sentences

sentences = preprocess(raw_text)
# Flatten all sentences into a single list of words
words = [w for s in sentences for w in s]

print(f"Sentences : {len(sentences)}")
print(f"Tokens    : {len(words)}")
print(f"Unique    : {len(set(words))}")
print()
print("First sentence:", sentences[0])

Sentences : 14
Tokens    : 188
Unique    : 129

First sentence: ['ultralytics', 'yolo', 'is', 'a', 'versatile', 'ai', 'framework', 'that', 'supports', 'multiple', 'computer', 'vision', 'tasks']


## Step 3: Build Vocabulary and Index Mappings

Each unique word gets a unique integer ID. We need two dictionaries:
- `word_to_index`: word → number
- `index_to_word`: number → word

In [ ]:
# Build vocabulary: all unique words (preserving order of appearance)
vocab = list(dict.fromkeys(words))  # deduplicate while keeping order
vocab_size = len(vocab)

# Create word ↔ index lookup tables
word_to_index = {word: i for i, word in enumerate(vocab)}
index_to_word = {i: word for word, i in word_to_index.items()}

print(f"Vocabulary size: {vocab_size}")
print()
print("Sample word → index mappings:")
for word in list(vocab)[:5]:
    print(f"  '{word}' → {word_to_index[word]}")

Vocabulary size: 129

Sample word → index mappings:
  'ultralytics' → 0
  'yolo' → 1
  'is' → 2
  'a' → 3
  'versatile' → 4


## Step 4: CBOW — Generate Training Pairs

For CBOW: each training sample is (context_words, target_word).

With `window_size=1`, for sentence `[a, b, c, d]`:
- context=[a, c], target=b
- context=[b, d], target=c

In [ ]:
# window_size = how many words to look on each side of the target
window_size = 1
cbow_training_data = []

for i in range(window_size, len(words) - window_size):
    context = []
    for j in range(-window_size, window_size + 1):
        if j != 0:  # skip the center word itself
            context.append(word_to_index[words[i + j]])
    target = word_to_index[words[i]]
    cbow_training_data.append((context, target))

print(f"CBOW training samples: {len(cbow_training_data)}")
print()
print("First 3 samples (context → target):")
for ctx, tgt in cbow_training_data[:3]:
    ctx_words = [index_to_word[i] for i in ctx]
    tgt_word = index_to_word[tgt]
    print(f"  context: {ctx_words} → target: '{tgt_word}'")

CBOW training samples: 186

First 3 samples (context → target):
  context: ['ultralytics', 'is'] → target: 'yolo'
  context: ['yolo', 'a'] → target: 'is'
  context: ['is', 'versatile'] → target: 'a'


## Step 5: CBOW — Initialize Weights

Two weight matrices:
- **W1** (vocab_size × embed_dim): the embedding matrix — each row is a word vector
- **W2** (embed_dim × vocab_size): output projection matrix

In [ ]:
np.random.seed(42)
embedding_dim = 5   # dimension of each word vector (small for learning purposes)

# W1: input weight matrix — row i = embedding of word i
W1_cbow = np.random.randn(vocab_size, embedding_dim) * 0.01

# W2: output weight matrix — maps hidden layer to vocabulary scores
W2_cbow = np.random.randn(embedding_dim, vocab_size) * 0.01

learning_rate = 0.01

print(f"W1 shape: {W1_cbow.shape}  (vocab_size x embedding_dim)")
print(f"W2 shape: {W2_cbow.shape}  (embedding_dim x vocab_size)")

W1 shape: (129, 5)  (vocab_size x embedding_dim)
W2 shape: (5, 129)  (embedding_dim x vocab_size)


## Step 6: Softmax Helper Function

Softmax converts raw scores (logits) into probabilities. We subtract `max(x)` first for numerical stability (avoids overflow in exp).

In [ ]:
def softmax(x):
    # Subtract max value before exp to prevent numerical overflow
    exp_x = np.exp(x - np.max(x))
    # Divide each value by the sum so all probabilities add to 1
    return exp_x / exp_x.sum()

## Step 7: CBOW — Training Loop

Forward pass: average context embeddings → compute scores → softmax  
Loss: cross-entropy  
Backward pass: gradient descent update on W1 and W2

In [ ]:
epochs = 500
print_every = 100
loss_history_cbow = []

for epoch in range(epochs):
    total_loss = 0

    for context_indices, target_index in cbow_training_data:

        # --- FORWARD PASS ---
        # Step 1: Get context word vectors and average them
        # h is the hidden layer: average embedding of all context words
        h = np.mean(W1_cbow[context_indices], axis=0)   # shape: (embed_dim,)

        # Step 2: Project hidden vector to vocabulary space
        u = np.dot(h, W2_cbow)                           # shape: (vocab_size,)

        # Step 3: Convert to probabilities using softmax
        y_pred = softmax(u)                              # shape: (vocab_size,)

        # --- LOSS ---
        # Cross-entropy loss: -log(probability of the correct target word)
        total_loss -= np.log(y_pred[target_index] + 1e-9)

        # --- BACKWARD PASS ---
        # Error = predicted probabilities minus one-hot of true target
        error = y_pred.copy()
        error[target_index] -= 1                         # shape: (vocab_size,)

        # Gradient for W2 (outer product of h and error)
        dW2 = np.outer(h, error)

        # Gradient for W1 (backpropagate error through W2, then into embeddings)
        dW1 = np.dot(W2_cbow, error) / len(context_indices)

        # --- WEIGHT UPDATE ---
        W2_cbow -= learning_rate * dW2
        for word_idx in context_indices:
            W1_cbow[word_idx] -= learning_rate * dW1

    loss_history_cbow.append(total_loss)
    if (epoch + 1) % print_every == 0:
        print(f"Epoch {epoch+1:>4}/{epochs}  |  Loss: {total_loss:.4f}")

print("\nCBOW training complete.")

Epoch  100/500  |  Loss: 903.6493
Epoch  200/500  |  Loss: 896.1770
Epoch  300/500  |  Loss: 752.6773
Epoch  400/500  |  Loss: 512.5141
Epoch  500/500  |  Loss: 349.7387

CBOW training complete.


## Step 8: CBOW — Learned Word Embeddings

The rows of W1 are the word vectors. Each word is now represented as a 5-dimensional vector.

In [ ]:
print("CBOW Word Embeddings (W1 rows):")
print(f"{'Word':<20}  Vector")
print("-" * 60)
for word in list(vocab)[:10]:
    vec = W1_cbow[word_to_index[word]]
    vals = ', '.join([f'{v:.4f}' for v in vec])
    print(f"{word:<20}  [{vals}]")

CBOW Word Embeddings (W1 rows):
Word                  Vector
------------------------------------------------------------
ultralytics           [-0.4582, -0.2295, 0.4482, -0.2124, -1.1027]
yolo                  [1.4810, -0.9498, 1.0009, 0.8074, -1.6394]
is                    [-2.0652, -0.7111, -0.2536, -2.3648, -0.9699]
a                     [-0.0021, -1.8764, 0.5094, 0.3979, -0.9543]
versatile             [-0.1551, 0.4504, -1.2685, -0.8412, 0.0338]
ai                    [-0.0984, 0.2649, 1.2680, -0.3344, -0.8988]
framework             [2.3214, 0.6600, 0.9288, 0.1925, 0.0661]
that                  [-0.3775, 0.4427, 1.2075, 0.1018, -1.1666]
supports              [0.7211, 0.1503, 0.5330, 0.6659, 0.1204]
multiple              [1.3309, -0.3832, 1.0434, 0.6278, -1.0912]


## Step 9: Skip-Gram — Generate Training Pairs

For Skip-Gram: each training sample is (center_word, one_context_word).
Note: each center word generates multiple training pairs — one per context word.

In [ ]:
skipgram_training_data = []

for i in range(window_size, len(words) - window_size):
    target_index = word_to_index[words[i]]  # center word

    for j in range(-window_size, window_size + 1):
        if j != 0:
            # Each (center, neighbor) pair is one training sample
            context_index = word_to_index[words[i + j]]
            skipgram_training_data.append((target_index, context_index))

print(f"Skip-Gram training samples: {len(skipgram_training_data)}")
print(f"(More samples than CBOW because each center word creates {window_size*2} pairs)")
print()
print("First 3 samples (center → context):")
for ctr, ctx in skipgram_training_data[:3]:
    print(f"  center: '{index_to_word[ctr]}' → context: '{index_to_word[ctx]}'")

Skip-Gram training samples: 372
(More samples than CBOW because each center word creates 2 pairs)

First 3 samples (center → context):
  center: 'yolo' → context: 'ultralytics'
  center: 'yolo' → context: 'is'
  center: 'is' → context: 'yolo'


## Step 10: Skip-Gram — Initialize Weights

In [ ]:
np.random.seed(42)

# Same structure as CBOW — two weight matrices
W1_sg = np.random.randn(vocab_size, embedding_dim) * 0.01
W2_sg = np.random.randn(embedding_dim, vocab_size) * 0.01

print(f"W1 shape: {W1_sg.shape}")
print(f"W2 shape: {W2_sg.shape}")

W1 shape: (129, 5)
W2 shape: (5, 129)


## Step 11: Skip-Gram — Training Loop

Forward pass: look up center word embedding → compute scores → softmax  
Backward: update using gradient descent

In [ ]:
loss_history_sg = []

for epoch in range(epochs):
    total_loss = 0

    for center_index, context_index in skipgram_training_data:

        # --- FORWARD PASS ---
        # Step 1: Get the center word's embedding (just a single row lookup)
        h = W1_sg[center_index]                      # shape: (embed_dim,)

        # Step 2: Project to vocabulary scores
        u = np.dot(h, W2_sg)                         # shape: (vocab_size,)

        # Step 3: Get probability distribution over all words
        y_pred = softmax(u)                          # shape: (vocab_size,)

        # --- LOSS ---
        total_loss -= np.log(y_pred[context_index] + 1e-9)

        # --- BACKWARD PASS ---
        error = y_pred.copy()
        error[context_index] -= 1                    # true label position gets -1

        # Gradient for W2
        dW2 = np.outer(h, error)

        # Gradient for W1 (only update the center word's row)
        dW1 = np.dot(W2_sg, error)

        # --- WEIGHT UPDATE ---
        W2_sg -= learning_rate * dW2
        W1_sg[center_index] -= learning_rate * dW1

    loss_history_sg.append(total_loss)
    if (epoch + 1) % print_every == 0:
        print(f"Epoch {epoch+1:>4}/{epochs}  |  Loss: {total_loss:.4f}")

print("\nSkip-Gram training complete.")

Epoch  100/500  |  Loss: 1793.1212
Epoch  200/500  |  Loss: 1202.6180
Epoch  300/500  |  Loss: 848.0355
Epoch  400/500  |  Loss: 705.5885
Epoch  500/500  |  Loss: 632.7747

Skip-Gram training complete.


## Step 12: Skip-Gram — Learned Word Embeddings

In [ ]:
print("Skip-Gram Word Embeddings (W1 rows):")
print(f"{'Word':<20}  Vector")
print("-" * 60)
for word in list(vocab)[:10]:
    vec = W1_sg[word_to_index[word]]
    vals = ', '.join([f'{v:.4f}' for v in vec])
    print(f"{word:<20}  [{vals}]")

Skip-Gram Word Embeddings (W1 rows):
Word                  Vector
------------------------------------------------------------
ultralytics           [0.0050, -0.0014, 0.0065, 0.0152, -0.0023]
yolo                  [2.0649, -1.9036, 0.7981, 1.9542, 0.0872]
is                    [-0.9754, -1.0158, 0.4020, -3.4616, 0.2188]
a                     [-0.3965, -1.8153, 0.1129, 0.3827, -0.1939]
versatile             [0.4234, 1.3972, -1.3240, -2.1346, 0.2668]
ai                    [-0.4774, -1.2927, 2.2671, -0.5357, -2.1752]
framework             [3.2640, 0.9975, 0.4974, -0.0816, 0.6418]
that                  [0.2372, -0.1595, 2.4614, 0.2306, -2.6608]
supports              [1.9419, 0.4297, 1.7949, 0.6159, 1.8223]
multiple              [1.8284, -0.9712, 1.6393, 0.4400, -3.1726]


## Step 13: Compare Loss Curves

Lower loss = the model is learning the word-context relationships better.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.plot(loss_history_cbow, label='CBOW', color='steelblue')
plt.plot(loss_history_sg,   label='Skip-Gram', color='darkorange')
plt.title("Training Loss: CBOW vs Skip-Gram")
plt.xlabel("Epoch")
plt.ylabel("Total Cross-Entropy Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/loss_curve.png', dpi=80)
plt.show()
print("Loss curve saved.")

Loss curve saved.


## Conclusion

- Both CBOW and Skip-Gram learn word embeddings as a side effect of training a prediction network.
- **CBOW** averages context vectors → faster, works well with frequent words.
- **Skip-Gram** creates one training example per context word → more data, better for rare words.
- The learned `W1` matrix IS the embedding table — each row is a word vector.
- On a large real corpus, semantically similar words end up with similar vectors automatically.